# Tutorial 4: Principal Component Analysis (PCA) from Scratch

---

## Learning Objectives

In this tutorial, you will learn how to implement PCA (Principal Component Analysis) from scratch. By the end of this notebook, you will understand:

1. How to represent data as a matrix
2. Computing feature means and mean centering
3. Understanding variance and covariance
4. Building the covariance matrix
5. Eigenvalue decomposition
6. Understanding eigenvectors and orthonormal vectors
7. Sorting eigenvalues and choosing principal components
8. Explained variance ratio
9. Projecting data onto principal components
10. PCA as coordinate rotation
11. PCA for visualization
12. Limitations of PCA (unsupervised nature)

---

## 1. Data Representation as a Matrix

The first step in PCA is representing our dataset as a matrix. Let's understand this concept.

### Theory

We represent the dataset as a matrix:

$$X \in \mathbb{R}^{n \times d}$$

Where:
- **n** = number of samples (rows)
- **d** = number of features (columns)

For the Iris dataset:
- $X \in \mathbb{R}^{150 \times 4}$
- 150 flower samples
- 4 features: sepal length, sepal width, petal length, petal width

### Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Load the Iris dataset
iris = load_iris()
X = iris.data  # Feature matrix
feature_names = iris.feature_names
target_names = iris.target_names

# Display dataset information
print("Dataset Shape:", X.shape)
print("Number of samples (n):", X.shape[0])
print("Number of features (d):", X.shape[1])
print("\nFeature names:", feature_names)
print("\nFirst 5 samples:")
print(X[:5])

---

## 2. Feature Mean

### Theory

The mean of each feature is calculated as:

$$\mu_j = \frac{1}{n}\sum_{i=1}^{n} x_{ij}$$

Where $j$ is the feature index.

**Why PCA needs it**: PCA assumes data is centered around zero. This is essential for computing covariance correctly.

### Code

In [ ]:
# Compute the mean of each feature
means = np.mean(X, axis=0)

print("Feature Means:")
for i, (name, mean) in enumerate(zip(feature_names, means)):
    print(f"  {name}: {mean:.4f}")

# Visualize the means
plt.figure(figsize=(10, 4))
plt.bar(range(len(feature_names)), means, color='steelblue', alpha=0.7)
plt.xticks(range(len(feature_names)), [name.replace(' (cm)', '') for name in feature_names], rotation=45)
plt.ylabel('Mean Value')
plt.title('Feature Means in Iris Dataset')
plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---

## 3. Mean Centering (Zero-Mean Data)

### Theory

Mean centering subtracts the mean from every feature:

$$X_{centered} = X - \mu$$

This ensures:

$$E[X] = 0$$

**Why PCA needs it**: Covariance measures spread around the mean. If data is not centered, covariance becomes incorrect.

### Code

In [ ]:
# Mean centering: subtract mean from each column
X_centered = X - means

# Verify that the mean is now zero
centered_means = np.mean(X_centered, axis=0)
print("Means after centering (should be ~0):")
print(centered_means)

# Compare distributions before and after centering
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X[:, 0], bins=20, alpha=0.7, color='steelblue')
axes[0].axvline(x=means[0], color='red', linestyle='--', label=f'Mean = {means[0]:.2f}')
axes[0].set_title('Before Centering')
axes[0].set_xlabel(feature_names[0])
axes[0].legend()

axes[1].hist(X_centered[:, 0], bins=20, alpha=0.7, color='green')
axes[1].axvline(x=0, color='red', linestyle='--', label='Mean = 0')
axes[1].set_title('After Centering')
axes[1].set_xlabel(f'{feature_names[0]} (centered)')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 4. Variance

### Theory

Variance measures how spread out the data is from the mean:

$$Var(x) = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \mu)^2$$

**Why PCA cares**: PCA finds directions of maximum variance. Higher variance = more information.

### Code

In [ ]:
# Compute variance for each feature
variances = np.var(X_centered, axis=0, ddof=1)  # ddof=1 for sample variance

print("Feature Variances (centered data):")
for name, var in zip(feature_names, variances):
    print(f"  {name}: {var:.4f}")

# Visualize variance
plt.figure(figsize=(10, 4))
plt.bar(range(len(feature_names)), variances, color='coral', alpha=0.7)
plt.xticks(range(len(feature_names)), [name.replace(' (cm)', '') for name in feature_names], rotation=45)
plt.ylabel('Variance')
plt.title('Feature Variances in Iris Dataset')
plt.tight_layout()
plt.show()

---

## 5. Covariance

### Theory

Covariance measures how two features move together:

$$Cov(x, y) = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \mu_x)(y_i - \mu_y)$$

| Value | Meaning |
|-------|---------|
Positive | Features move together |
Negative | Features move opposite |
Zero | Features are independent |

### Code

In [ ]:
# Compute covariance between first two features
cov_sepal_length_sepal_width = np.cov(X_centered[:, 0], X_centered[:, 1])[0, 1]
print(f"Covariance between sepal length and sepal width: {cov_sepal_length_sepal_width:.4f}")

# Compute covariance between petal length and petal width
cov_petal_length_petal_width = np.cov(X_centered[:, 2], X_centered[:, 3])[0, 1]
print(f"Covariance between petal length and petal width: {cov_petal_length_petal_width:.4f}")

# Visualize relationship between features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5)
axes[0].set_xlabel('Sepal Length (centered)')
axes[0].set_ylabel('Sepal Width (centered)')
axes[0].set_title(f'Covariance = {cov_sepal_length_sepal_width:.2f}')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.3)

axes[1].scatter(X_centered[:, 2], X_centered[:, 3], alpha=0.5, color='green')
axes[1].set_xlabel('Petal Length (centered)')
axes[1].set_ylabel('Petal Width (centered)')
axes[1].set_title(f'Covariance = {cov_petal_length_petal_width:.2f}')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.3)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. Covariance Matrix

### Theory

The covariance matrix captures relationships between all pairs of features:

$$C = \frac{1}{n-1} X_{centered}^T X_{centered}$$

Where $C \in \mathbb{R}^{d \times d}$

For Iris (4 features): $C \in \mathbb{R}^{4 \times 4}$

**Structure:**
- **Diagonal entries**: Variances of each feature
- **Off-diagonal entries**: Covariances between features

### Code

In [ ]:
# Compute covariance matrix manually
n = X_centered.shape[0]
cov_matrix = (X_centered.T @ X_centered) / (n - 1)

print("Covariance Matrix (4x4):")
print(np.round(cov_matrix, 4))

# Verify with numpy
cov_matrix_np = np.cov(X_centered.T)
print("\nUsing np.cov (should be same):")
print(np.allclose(cov_matrix, cov_matrix_np))

# Visualize covariance matrix
plt.figure(figsize=(8, 6))
plt.imshow(cov_matrix, cmap='coolwarm', aspect='auto')
plt.colorbar(label='Covariance')
plt.xticks(range(4), [name.replace(' (cm)', '') for name in feature_names], rotation=45)
plt.yticks(range(4), [name.replace(' (cm)', '') for name in feature_names])
plt.title('Covariance Matrix Heatmap')

# Add values to heatmap
for i in range(4):
    for j in range(4):
        plt.text(j, i, f'{cov_matrix[i, j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---

## 7. Eigenvalues and Eigenvectors

### Theory

Eigenvalues and eigenvectors are fundamental to PCA:

If $Cv = \lambda v$, then:
- $v$ = eigenvector (direction)
- $\lambda$ = eigenvalue (importance/ variance)

**In PCA:**
- Each eigenvector represents one principal component (a new direction)
- Each eigenvalue represents the variance captured by that direction

### Code

In [ ]:
# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Eigenvalues should be real for symmetric matrix
eigenvalues = np.real(eigenvalues)
eigenvectors = np.real(eigenvectors)

print("Eigenvalues:")
for i, ev in enumerate(eigenvalues):
    print(f"  λ{i+1} = {ev:.4f}")

print("\nEigenvectors (each column is an eigenvector):")
print(np.round(eigenvectors, 4))

---

## 8. Understanding Eigenvectors

### Theory

Each eigenvector represents a new coordinate direction - these are the principal components. They form an orthogonal basis that captures the maximum variance in the data.

### Code

In [ ]:
# Visualize eigenvectors as arrows on the feature space
# Use first two principal components (we'll compute them soon)

# First, let's understand eigenvectors conceptually
print("Eigenvector properties:")
print("="*50)

for i in range(4):
    print(f"\nPrincipal Component {i+1} (PC{i+1}):")
    print(f"  Eigenvalue: {eigenvalues[i]:.4f}")
    print(f"  Eigenvector: {eigenvectors[:, i]}")

# Visualize the eigenvectors as directions in 2D space
# For visualization, we'll project to 2D
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Plot centered data
ax.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.3, s=20, label='Data points')

# Scale eigenvectors for visibility
scale = 3
colors = ['red', 'green', 'blue', 'purple']

for i in range(2):  # First 2 eigenvectors
    ax.arrow(0, 0, eigenvectors[0, i]*scale*np.sqrt(eigenvalues[i]), 
             eigenvectors[1, i]*scale*np.sqrt(eigenvalues[i]),
             head_width=0.15, head_length=0.1, fc=colors[i], ec=colors[i], linewidth=2)
    ax.text(eigenvectors[0, i]*scale*np.sqrt(eigenvalues[i])*1.1,
            eigenvectors[1, i]*scale*np.sqrt(eigenvalues[i])*1.1,
            f'PC{i+1}', fontsize=12, color=colors[i], fontweight='bold')

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Sepal Length (centered)')
ax.set_ylabel('Sepal Width (centered)')
ax.set_title('Eigenvectors as Directions of Maximum Variance')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---

## 9. Orthonormal Vectors

### Theory

Eigenvectors in PCA are orthonormal (orthogonal + unit length):

$$v_i^T v_j = \begin{cases} 1 & i=j \\ 0 & i \neq j \end{cases}$$

This means:
- They are perpendicular to each other (orthogonal)
- Each has length 1 (unit vectors)

### Code

In [ ]:
# Verify orthonormality
print("Orthonormality Check:")
print("="*50)

# Check dot products (orthogonal = dot product is 0)
print("\nDot products between eigenvectors:")
for i in range(4):
    for j in range(4):
        dot_product = np.dot(eigenvectors[:, i], eigenvectors[:, j])
        if i == j:
            print(f"  v{i+1} · v{j+1} = {dot_product:.4f} (should be 1)")
        else:
            print(f"  v{i+1} · v{j+1} = {dot_product:.4f} (should be 0)")

# Check vector norms (should all be 1)
print("\nVector norms (should all be 1):")
for i in range(4):
    norm = np.linalg.norm(eigenvectors[:, i])
    print(f"  ||v{i+1}|| = {norm:.4f}")

---

## 10. Sorting Eigenvalues

### Theory

We sort eigenvalues in descending order (largest first):

$$\lambda_1 \ge \lambda_2 \ge \lambda_3 \ge \lambda_4$$

This ensures the most important principal components come first.

### Code

In [ ]:
# Sort eigenvalues in descending order
sorted_indices = np.argsort(eigenvalues)[::-1]
eigenvalues_sorted = eigenvalues[sorted_indices]
eigenvectors_sorted = eigenvectors[:, sorted_indices]

print("Sorted Eigenvalues (descending):")
for i, ev in enumerate(eigenvalues_sorted):
    print(f"  λ{i+1} = {ev:.4f}")

print("\nSorted Eigenvectors (corresponding to sorted eigenvalues):")
print(np.round(eigenvectors_sorted, 4))

---

## 11. Explained Variance Ratio

### Theory

The explained variance ratio tells us what fraction of total variance each PC captures:

$$EVR_i = \frac{\lambda_i}{\sum \lambda}$$

Example:
| PC | Variance |
|----|----------|
| PC1 | 72% |
| PC2 | 23% |
| PC3 | 4% |
| PC4 | 1% |

### Code

In [ ]:
# Calculate explained variance ratio
total_variance = np.sum(eigenvalues_sorted)
explained_variance_ratio = eigenvalues_sorted / total_variance

print("Explained Variance Ratio:")
for i, evr in enumerate(explained_variance_ratio):
    print(f"  PC{i+1}: {evr*100:.2f}%")

# Cumulative explained variance
cumulative_variance = np.cumsum(explained_variance_ratio)

print("\nCumulative Explained Variance:")
for i, cv in enumerate(cumulative_variance):
    print(f"  PC1 to PC{i+1}: {cv*100:.2f}%")

# Visualize explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 5), explained_variance_ratio * 100, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Explained Variance per Component')
axes[0].set_xticks(range(1, 5))

axes[1].plot(range(1, 5), cumulative_variance * 100, 'o-', color='green', linewidth=2)
axes[1].axhline(y=95, color='red', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xticks(range(1, 5))
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 12. Choosing Number of Components (95% Rule)

### Theory

We choose the smallest number of components $d$ such that:

$$ \sum_{i=1}^{d} EVR_i \ge 0.95 $$

This captures at least 95% of the variance while reducing dimensionality.

### Code

In [ ]:
# Find number of components needed for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print(f"Number of components for 95% variance: {n_components_95}")
print(f"Cumulative variance with {n_components_95} components: {cumulative_variance[n_components_95-1]*100:.2f}%")

# What if we wanted 99%?
n_components_99 = np.argmax(cumulative_variance >= 0.99) + 1
print(f"\nNumber of components for 99% variance: {n_components_99}")
print(f"Cumulative variance with {n_components_99} components: {cumulative_variance[n_components_99-1]*100:.2f}%")

---

## 13. Projection onto Principal Components

### Theory

To transform data onto the new PCA basis:

$$Z = X_{centered} W$$

Where:
- $W$ = eigenvector matrix (principal components)
- $Z$ = transformed data in PCA space

This is essentially a rotation of the coordinate system.

### Code

In [ ]:
# Project data onto principal components
W = eigenvectors_sorted  # Each column is a principal component

# Transform to PCA space
X_pca = X_centered @ W

print("Original data shape:", X.shape)
print("Transformed data shape (PCA):", X_pca.shape)

print("\nFirst 5 samples in PCA space:")
print(X_pca[:5])

# Verify: variance in each PCA direction should match eigenvalues
pca_variances = np.var(X_pca, axis=0, ddof=1)
print("\nVariance in each PCA direction:", np.round(pca_variances, 4))
print("Original eigenvalues:", np.round(eigenvalues_sorted, 4))

---

## 14. PCA as Coordinate Rotation

### Theory

PCA can be understood as rotating the coordinate system so that:
- First axis → direction of maximum variance
- Second axis → second maximum variance
- And so on...

This is called a "change of basis" or "coordinate rotation."

### Code

In [ ]:
# Visualize the coordinate rotation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original data (first two features)
scatter1 = axes[0].scatter(X[:, 0], X[:, 1], c=iris.target, cmap='viridis', alpha=0.6)
axes[0].set_xlabel('Sepal Length (cm)')
axes[0].set_ylabel('Sepal Width (cm)')
axes[0].set_title('Original Feature Space')
plt.colorbar(scatter1, ax=axes[0], label='Class')

# PCA transformed data (first two PCs)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=iris.target, cmap='viridis', alpha=0.6)
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')
axes[1].set_title('PCA Transformed Space (Rotation)')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.3)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Class')

plt.tight_layout()
plt.show()

print("Notice how the data is now spread along new axes that maximize variance!")

---

## 15. Covariance in PCA Basis

### Theory

After transforming to PCA space, the covariance matrix becomes diagonal:

$$Cov(Z) = \begin{bmatrix} \lambda_1 & 0 & 0 & 0 \\ 0 & \lambda_2 & 0 & 0 \\ 0 & 0 & \lambda_3 & 0 \\ 0 & 0 & 0 & \lambda_4 \end{bmatrix}$$

This means all features become uncorrelated!

### Code

In [ ]:
# Compute covariance in PCA space
cov_pca = np.cov(X_pca.T)

print("Covariance Matrix in PCA space:")
print(np.round(cov_pca, 4))

print("\nNotice: Off-diagonal elements are ~0! Features are uncorrelated.")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(cov_pca, cmap='coolwarm', aspect='auto')
plt.colorbar(label='Covariance')
plt.xticks(range(4), [f'PC{i+1}' for i in range(4)])
plt.yticks(range(4), [f'PC{i+1}' for i in range(4)])
plt.title('Covariance Matrix in PCA Space (Diagonal!)')

for i in range(4):
    for j in range(4):
        plt.text(j, i, f'{cov_pca[i, j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---

## 16. PCA for Visualization

### Theory

One common use of PCA is to reduce data to 2D for visualization:

$$Z \in \mathbb{R}^{150 \times 2}$$

Then we can create scatter plots colored by class.

### Code

In [ ]:
# PCA for visualization (reduce to 2D)
X_pca_2d = X_pca[:, :2]

# Create visualization
plt.figure(figsize=(10, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
markers = ['o', 's', '^']

for i, (color, marker, name) in enumerate(zip(colors, markers, target_names)):
    mask = iris.target == i
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], 
                c=color, marker=marker, label=name, s=60, alpha=0.7)

plt.xlabel(f'PC1 ({explained_variance_ratio[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({explained_variance_ratio[1]*100:.1f}% variance)')
plt.title('PCA Visualization of Iris Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nWith just 2 components, we capture {cumulative_variance[1]*100:.1f}% of the variance!")

---

## 17. Important PCA Limitation

### Theory

**PCA is an unsupervised method** - it does NOT use class labels!

This has important implications:
- PCA maximizes variance, NOT class separation
- It may not produce good class separation
- The directions of maximum variance may not correspond to directions that separate classes

### Code - Demonstration of Limitation

In [ ]:
# Demonstrate PCA limitation

print("="*60)
print("IMPORTANT: PCA LIMITATION")
print("="*60)

print("""
PCA is UNSUPERVISED - it does NOT use class labels!

It only finds directions of MAXIMUM VARIANCE.
These directions may NOT separate the classes well.

This is because:
1. PCA maximizes overall variance
2. It doesn't care about class labels
3. The class-separating directions might have less variance

In our Iris example:
- PC1 captures 72.96% of variance
- But this doesn't mean classes are well separated!

For better class separation, consider:
- Linear Discriminant Analysis (LDA) - supervised!
- Other supervised dimensionality reduction methods
""")

# Show class separation in PCA space
print("\nClass separation analysis in PCA space:")
for i, name in enumerate(target_names):
    mask = iris.target == i
    pc1_mean = np.mean(X_pca_2d[mask, 0])
    pc2_mean = np.mean(X_pca_2d[mask, 1])
    print(f"  {name}: PC1 mean = {pc1_mean:.2f}, PC2 mean = {pc2_mean:.2f}")

---

## Complete PCA Implementation Summary

Let's put everything together into a complete PCA implementation:

In [ ]:
def pca_from_scratch(X, n_components=None, variance_threshold=0.95):
    """
    Implement PCA from scratch.
    
    Parameters:
    -----------
    X : np.array
        Data matrix of shape (n_samples, n_features)
    n_components : int, optional
        Number of components to keep
    variance_threshold : float, optional
        If n_components not specified, keep components until 
        this fraction of variance is explained
    Returns:
    --------
    X_transformed : np.array
        Transformed data
    components : np.array
        Principal components (eigenvectors)
    explained_variance : np.array
        Explained variance for each component
    """
    # Step 1: Mean centering
    means = np.mean(X, axis=0)
    X_centered = X - means
    
    # Step 2: Compute covariance matrix
    n = X_centered.shape[0]
    cov_matrix = (X_centered.T @ X_centered) / (n - 1)
    
    # Step 3: Eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    eigenvalues = np.real(eigenvalues)
    eigenvectors = np.real(eigenvectors)
    
    # Step 4: Sort by eigenvalues (descending)
    sorted_indices = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_indices]
    eigenvectors = eigenvectors[:, sorted_indices]
    
    # Step 5: Determine number of components
    total_variance = np.sum(eigenvalues)
    explained_variance_ratio = eigenvalues / total_variance
    cumulative_variance = np.cumsum(explained_variance_ratio)
    
    if n_components is None:
        n_components = np.argmax(cumulative_variance >= variance_threshold) + 1
    
    # Step 6: Select top n_components
    components = eigenvectors[:, :n_components]
    explained_variance = eigenvalues[:n_components]
    
    # Step 7: Transform data
    X_transformed = X_centered @ components
    
    return X_transformed, components, explained_variance, explained_variance_ratio[:n_components]

# Test our implementation
print("Testing our PCA implementation from scratch!")
print("="*50)

X_pca_ours, components, exp_var, exp_var_ratio = pca_from_scratch(X)

print(f"Transformed shape: {X_pca_ours.shape}")
print(f"Number of components (95% rule): {components.shape[1]}")
print(f"Explained variance ratios: {exp_var_ratio * 100}")

# Compare with sklearn
from sklearn.decomposition import PCA
pca_sklearn = PCA(n_components=0.95)
X_pca_sklearn = pca_sklearn.fit_transform(X)

print("\n" + "="*50)
print("Comparison with sklearn:")
print(f"Our implementation: {X_pca_ours.shape}, sklearn: {X_pca_sklearn.shape}")
print(f"Our explained variance: {exp_var_ratio * 100}")
print(f"Sklearn explained variance: {pca_sklearn.explained_variance_ratio_ * 100}")

---

## Summary: 17 Concepts in PCA

You have now learned all 17 key concepts in PCA:

1. ✅ **Dataset as matrix** - Representing data as $X \in \mathbb{R}^{n \times d}$
2. ✅ **Feature mean** - Computing $\mu_j = \frac{1}{n}\sum x_{ij}$
3. ✅ **Mean centering** - Subtracting mean: $X_{centered} = X - \mu$
4. ✅ **Variance** - $Var(x) = \frac{1}{n-1}\sum (x_i - \mu)^2$
5. ✅ **Covariance** - Measuring joint variability
6. ✅ **Covariance matrix** - $C = \frac{1}{n-1} X^T X$
7. ✅ **Eigenvalues** - Representing variance in each direction
8. ✅ **Eigenvectors** - Representing new coordinate directions
9. ✅ **Orthonormal vectors** - Verifying $v_i^T v_j = \delta_{ij}$
10. ✅ **Sorting eigenvalues** - $\lambda_1 \ge \lambda_2 \ge ...$
11. ✅ **Explained variance ratio** - $EVR_i = \frac{\lambda_i}{\sum \lambda}$
12. ✅ **Choosing components** - 95% threshold rule
13. ✅ **Projection** - $Z = X_{centered} W$
14. ✅ **PCA as rotation** - Change of basis
15. ✅ **Covariance diagonalization** - Features become uncorrelated
16. ✅ **PCA for visualization** - 2D scatter plots
17. ✅ **PCA limitation** - Unsupervised (doesn't use class labels)

---

## Python Skills Used
- NumPy: matrix operations, mean, transpose, eigen decomposition
- Matplotlib: scatter plots, visualization
- Indexing and slicing
- Broadcasting

---

## Exercises for Students

1. Apply PCA to a different dataset (e.g., wine dataset from sklearn)
2. Compare results with sklearn's PCA implementation
3. Visualize the data in 3D using the first 3 principal components
4. Explain in your own words why PCA might not separate classes well
5. Research: What is the relationship between PCA and SVD (Singular Value Decomposition)?

---

**End of Tutorial 4**